# Baseball Reference scrape workflow

This notebook is a handoff for porting the existing Baseball Reference workflow in this repo to another project.

It covers two pieces:

1. how to build a season-wide list of Baseball Reference `game_id` values for all 30 MLB teams
2. how to fetch and parse box score pages for those `game_id` values

The logic below mirrors the current project scripts:

- `scripts/mlb/scrape_baseball_reference_schedule.py`
- `scripts/mlb/build_br_game_ids.py`
- `scripts/mlb/scrape_br_play_by_play.py`

Key detail: Baseball Reference pages often keep tables inside HTML comments, so plain `pandas.read_html(url)` is not enough.

## Workflow notes

- Team schedule pages live at `https://www.baseball-reference.com/teams/{TEAM}/{SEASON}-schedule-scores.shtml#all_team_schedule`
- Box score pages live at `https://www.baseball-reference.com/boxes/{HOME_BR_CODE}/{game_id}.shtml`
- A BR `game_id` is `home_team_br_code + YYYYMMDD + game_number`
- `game_number` is usually `0`; doubleheaders use `1` and `2`
- If you scrape schedules for all 30 teams, every game appears twice, so you must deduplicate on `game_id`
- BR uses some home-team codes that differ from standard MLB abbreviations, for example `CHC -> CHN`, `CHW -> CHA`, `LAD -> LAN`, `SDP -> SDN`, `SFG -> SFN`, `STL -> SLN`, `TBR -> TBA`, `WSN -> WAS`, `KCR -> KCA`
- In this notebook, page fetches use `requests` for portability. In the current repo, `scripts/mlb/scrape_br_play_by_play.py` adds Selenium retry logic; if another project sees blocking or partial page loads, port that fallback too.

In [ ]:
from __future__ import annotations

import io
import random
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment

SEASON = 2025
TEAM_CODES = [
    "ARI", "ATL", "BAL", "BOS", "CHC", "CHW", "CIN", "CLE", "COL", "DET",
    "HOU", "KCR", "LAA", "LAD", "MIA", "MIL", "MIN", "NYM", "NYY", "ATH",
    "PHI", "PIT", "SDP", "SEA", "SFG", "STL", "TBR", "TEX", "TOR", "WSN",
]

BR_TEAM_CODES = {
    "ANA": "ANA",
    "ARI": "ARI",
    "ATH": "ATH",
    "ATL": "ATL",
    "BAL": "BAL",
    "BOS": "BOS",
    "CHC": "CHN",
    "CHW": "CHA",
    "CIN": "CIN",
    "CLE": "CLE",
    "COL": "COL",
    "DET": "DET",
    "HOU": "HOU",
    "KCR": "KCA",
    "LAA": "ANA",
    "LAD": "LAN",
    "MIA": "MIA",
    "MIL": "MIL",
    "MIN": "MIN",
    "NYM": "NYN",
    "NYY": "NYA",
    "OAK": "OAK",
    "ATH": "ATH",
    "PHI": "PHI",
    "PIT": "PIT",
    "SDP": "SDN",
    "SEA": "SEA",
    "SFG": "SFN",
    "STL": "SLN",
    "TBR": "TBA",
    "TEX": "TEX",
    "TOR": "TOR",
    "WSN": "WAS",
}

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/127.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.baseball-reference.com/",
}

OUT_DIR = Path("br_handoff_outputs")
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
def flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join(str(part) for part in col if str(part) not in {"", "nan"}).strip("_")
            for col in df.columns
        ]
    else:
        df.columns = [str(col).strip() for col in df.columns]
    unnamed = [col for col in df.columns if col.startswith("Unnamed:")]
    if unnamed:
        df = df.drop(columns=unnamed)
    return df


def parse_table_from_html(html: str, table_id: str) -> pd.DataFrame:
    soup = BeautifulSoup(html, "lxml")

    direct = soup.find(id=table_id)
    if direct is not None:
        try:
            return flatten_columns(pd.read_html(io.StringIO(str(direct)))[0])
        except Exception:
            pass

    wrapper = soup.find(id=f"all_{table_id}")
    if wrapper is not None:
        for node in wrapper.descendants:
            if isinstance(node, Comment) and "<table" in node:
                inner = BeautifulSoup(str(node), "lxml").find(id=table_id)
                if inner is not None:
                    return flatten_columns(pd.read_html(io.StringIO(str(inner)))[0])

    for node in soup.find_all(string=lambda text: isinstance(text, Comment)):
        if table_id not in node:
            continue
        inner = BeautifulSoup(str(node), "lxml").find(id=table_id)
        if inner is not None:
            return flatten_columns(pd.read_html(io.StringIO(str(inner)))[0])

    raise ValueError(f"Could not find table '{table_id}' in page source")


def list_table_ids(html: str) -> list[str]:
    soup = BeautifulSoup(html, "lxml")
    ids = {tag.get("id") for tag in soup.find_all(id=True) if tag.name == "table"}
    for node in soup.find_all(string=lambda text: isinstance(text, Comment)):
        if "<table" not in node:
            continue
        inner = BeautifulSoup(str(node), "lxml")
        ids.update(tag.get("id") for tag in inner.find_all("table", id=True))
    return sorted(x for x in ids if x)


def fetch_html(url: str, pause_seconds: float = 0.0) -> str:
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    if pause_seconds > 0:
        time.sleep(pause_seconds)
    return response.text

## Part 1: build season-wide `game_id` values

The existing repo builds `game_id` values from a single team schedule. To do this for a whole season:

- fetch the schedule page for each of the 30 teams
- parse the `team_schedule` table
- compute the BR home-team code from `dest` and `Opp`/`Tm`
- strip any doubleheader suffix like `" (2)"` from the date before parsing
- use the suffix as the final digit of the `game_id`
- concatenate all teams together and deduplicate on `game_id`

Because each game is on both teams' schedules, the dedupe step is required.

In [ ]:
def scrape_team_schedule(team_code: str, season: int) -> pd.DataFrame:
    url = f"https://www.baseball-reference.com/teams/{team_code}/{season}-schedule-scores.shtml#all_team_schedule"
    html = fetch_html(url, pause_seconds=0.5)
    df = parse_table_from_html(html, "team_schedule")
    df["schedule_team"] = team_code
    df["source_url"] = url
    return df


def build_game_ids_from_schedule(df: pd.DataFrame, season: int) -> pd.DataFrame:
    work = df.copy()
    work["dest"] = work["dest"].fillna("")
    work = work[work["Date"].notna()].copy()
    work = work[~work["Date"].astype(str).str.contains("Postponed", na=False)].copy()

    work["date_clean"] = work["Date"].astype(str).str.replace(r"\s+\((\d+)\)$", "", regex=True)
    work["game_number"] = work["Date"].astype(str).str.extract(r"\((\d+)\)$", expand=False).fillna("0")
    work["home_team"] = work.apply(
        lambda row: row["Opp"].strip() if row["dest"] == "@" else row["Tm"].strip(),
        axis=1,
    )
    work["br_home_code"] = work["home_team"].map(BR_TEAM_CODES)

    missing = sorted(work.loc[work["br_home_code"].isna(), "home_team"].dropna().unique())
    if missing:
        raise ValueError(f"Missing BR code mapping for: {missing}")

    work["game_date"] = pd.to_datetime(
        work["date_clean"].str.strip() + f" {season}",
        format="%A %b %d %Y",
        errors="coerce",
    )
    work = work[work["game_date"].notna()].copy()

    work["game_id"] = (
        work["br_home_code"]
        + work["game_date"].dt.strftime("%Y%m%d")
        + work["game_number"]
    )

    return work[[
        "game_id", "game_date", "game_number", "Tm", "Opp", "dest", "home_team",
        "br_home_code", "schedule_team", "source_url",
    ]]


all_schedule_frames = []
for team_code in TEAM_CODES:
    print(f"scraping {team_code} schedule")
    team_schedule = scrape_team_schedule(team_code, SEASON)
    all_schedule_frames.append(build_game_ids_from_schedule(team_schedule, SEASON))
    time.sleep(1.0 + random.uniform(0.2, 0.8))

all_team_games = pd.concat(all_schedule_frames, ignore_index=True)
all_game_ids = (
    all_team_games
    .sort_values(["game_date", "game_id", "schedule_team"])
    .drop_duplicates(subset=["game_id"])
    .reset_index(drop=True)
)

all_team_games.to_csv(OUT_DIR / f"br_all_teams_schedule_rows_{SEASON}.csv", index=False)
all_game_ids.to_csv(OUT_DIR / f"br_game_ids_{SEASON}.csv", index=False)

print(f"raw rows: {len(all_team_games):,}")
print(f"unique game_ids: {len(all_game_ids):,}")
all_game_ids.head()

## Part 2: scrape box score pages from `game_id`

Once you have `game_id`, the URL pattern is deterministic:

`https://www.baseball-reference.com/boxes/{game_id[:3]}/{game_id}.shtml`

The safest pattern is:

- fetch the page HTML
- inspect available table ids
- parse the tables you care about using the same comment-aware parser

That approach avoids hard-wiring assumptions about which tables are commented out on a given page.

In [ ]:
def boxscore_url(game_id: str) -> str:
    return f"https://www.baseball-reference.com/boxes/{game_id[:3]}/{game_id}.shtml"


def scrape_boxscore_tables(game_id: str, table_ids: list[str] | None = None) -> dict[str, pd.DataFrame]:
    url = boxscore_url(game_id)
    html = fetch_html(url, pause_seconds=0.5)

    if table_ids is None:
        table_ids = list_table_ids(html)

    out: dict[str, pd.DataFrame] = {}
    for table_id in table_ids:
        try:
            df = parse_table_from_html(html, table_id)
            df.insert(0, "table_id", table_id)
            df.insert(0, "source_url", url)
            df.insert(0, "game_id", game_id)
            out[table_id] = df
        except Exception as exc:
            print(f"skipping {table_id}: {exc}")
    return out


sample_game_id = all_game_ids.loc[0, "game_id"]
sample_url = boxscore_url(sample_game_id)
sample_html = fetch_html(sample_url)

available_table_ids = list_table_ids(sample_html)
print(sample_game_id)
print(sample_url)
available_table_ids[:25]

In [ ]:
# Replace these with the exact table ids you want after inspecting available_table_ids.
# Common choices are team batting, team pitching, line score, and play-by-play related tables.
TARGET_TABLE_IDS = []

if TARGET_TABLE_IDS:
    target_out_dir = OUT_DIR / f"boxscore_tables_{SEASON}"
    target_out_dir.mkdir(exist_ok=True)

    for i, game_id in enumerate(all_game_ids["game_id"], start=1):
        print(f"[{i}/{len(all_game_ids)}] scraping {game_id}")
        tables = scrape_boxscore_tables(game_id, TARGET_TABLE_IDS)
        for table_id, df in tables.items():
            safe_table_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", table_id)
            df.to_csv(target_out_dir / f"{game_id}__{safe_table_id}.csv", index=False)
        time.sleep(1.5 + random.uniform(0.5, 1.5))
else:
    print("Set TARGET_TABLE_IDS after you inspect the first page's available_table_ids output.")

## Hand-off summary

If another agent is implementing this in a different repo, the minimum pieces to carry over are:

- the `BR_TEAM_CODES` mapping
- the comment-aware `parse_table_from_html()` helper
- the season-wide schedule scrape loop across all 30 teams
- `game_id` construction based on home team, date, and doubleheader suffix
- deduplication of `game_id` after combining team schedules
- the box score URL pattern `boxes/{game_id[:3]}/{game_id}.shtml`

If they specifically want play-by-play rather than general box score tables, use the same page URL with `table_id="play_by_play"`, which matches the existing `scripts/mlb/scrape_br_play_by_play.py` logic.